## Практическая работа 2 — LSA (C#)

- Предобработка текста
- TF-IDF (по общему корпусу)
- Снижение размерности (PCA как LSA)
- L2-нормализация
- Логистическая регрессия и метрики


In [ ]:
#r "nuget: Microsoft.Data.Analysis, 0.22.2"
#r "nuget: Microsoft.ML, 3.0.1"
#r "nuget: Microsoft.ML.Mkl.Components, 3.0.1"
#r "nuget: ScottPlot, 5.0.56"
#r "nuget: System.Text.Encoding.CodePages, 8.0.0"


In [ ]:
using System;
using System.IO;
using System.Text;
using System.Text.RegularExpressions;
using System.Collections.Generic;
using System.Linq;

using Microsoft.Data.Analysis;
using Microsoft.ML;
using Microsoft.ML.Data;
using Microsoft.ML.Transforms.Text;

using ScottPlot;
using Microsoft.DotNet.Interactive.Formatting;

Encoding.RegisterProvider(CodePagesEncodingProvider.Instance);

Formatter.Register(typeof(ScottPlot.Plot),
    (obj, writer) => writer.Write(((ScottPlot.Plot)obj).GetPngHtml(900, 520)),
    HtmlFormatter.MimeType);

var mlContext = new MLContext(seed: 42);


### 1) Загрузка и фильтрация данных (оставляем только text/sentiment, убираем neutral)

In [ ]:
DataFrame LoadDf(string path, string encoding = "windows-1251")
{
    using var fs = File.OpenRead(path);
    return DataFrame.LoadCsv(fs, separator: ',', header: true, guessRows: 50_000,
                             addIndexColumn: false, encoding: Encoding.GetEncoding(encoding));
}

var dfTrainRaw = LoadDf("../data/train.csv");
var dfTestRaw  = LoadDf("../data/test.csv");

// Оставляем только нужные колонки
var df_train = new DataFrame(dfTrainRaw.Columns["text"].Clone(), dfTrainRaw.Columns["sentiment"].Clone());
var df_test  = new DataFrame(dfTestRaw.Columns["text"].Clone(),  dfTestRaw.Columns["sentiment"].Clone());

var tr_text = (StringDataFrameColumn)df_train.Columns["text"];
var tr_sent = (StringDataFrameColumn)df_train.Columns["sentiment"];
var te_text = (StringDataFrameColumn)df_test.Columns["text"];
var te_sent = (StringDataFrameColumn)df_test.Columns["sentiment"];

// Фильтры: not null и sentiment != neutral
df_train = df_train.Filter(tr_text.ElementwiseIsNotNull())
                   .Filter(tr_sent.ElementwiseIsNotNull())
                   .Filter(tr_sent.ElementwiseNotEquals("neutral"));
df_test  = df_test .Filter(te_text.ElementwiseIsNotNull())
                   .Filter(te_sent.ElementwiseIsNotNull())
                   .Filter(te_sent.ElementwiseNotEquals("neutral"));

Console.WriteLine($"train shape: ({df_train.Rows.Count}, {df_train.Columns.Count})");
Console.WriteLine($"test  shape: ({df_test.Rows.Count},  {df_test.Columns.Count})");
Console.WriteLine(df_train.Head(5));


### 2) Предобработка текста (lower, чистка URL/HTML/пунктуации/чисел, простая фильтрация пустых)

In [ ]:
string CleanText(string text)
{
    if (string.IsNullOrWhiteSpace(text)) return string.Empty;
    var t = text.ToLowerInvariant();
    t = Regex.Replace(t, @"\[.*?\]", "");
    t = Regex.Replace(t, @"https?://\S+|www\.\S+", "");
    t = Regex.Replace(t, @"<.*?>+", "");
    t = Regex.Replace(t, @"\n", " " );
    t = Regex.Replace(t, @"\w*\d\w*", " " );
    t = Regex.Replace(t, @"[^a-z]+", " " ); // только буквы
    t = Regex.Replace(t, @"\s+", " " ).Trim();
    return t;
}

bool ToLabel(string s) => s == "positive";

public class DataExample { public string OriginalText { get; set; } = ""; public string Text { get; set; } = ""; public bool Label { get; set; } }

List<DataExample> ToExamples(DataFrame df)
{
    var list = new List<DataExample>(df.Rows.Count());
    foreach (var r in df.Rows)
    {
        var orig = (string)r["text"];
        var txt = CleanText(orig);
        if (string.IsNullOrWhiteSpace(txt)) continue;
        var lab = ToLabel((string)r["sentiment"]);
        list.Add(new DataExample { OriginalText = orig, Text = txt, Label = lab });
    }
    return list;
}

var trainList = ToExamples(df_train);
var testList  = ToExamples(df_test);
Console.WriteLine($"after preprocess: train={trainList.Count}, test={testList.Count}");


### 3) TF-IDF по общему корпусу (train+test) и оценка размерности/разреженности

In [ ]:
// IDataView
var fullData  = mlContext.Data.LoadFromEnumerable(trainList.Concat(testList));
var trainData = mlContext.Data.LoadFromEnumerable(trainList);
var testData  = mlContext.Data.LoadFromEnumerable(testList);

// Нормализация → токенизация → стоп-слова → TF-IDF
var textPipeline =
    mlContext.Transforms.Text.NormalizeText(outputColumnName: "TextNorm", inputColumnName: "Text",
        caseMode: TextNormalizingEstimator.CaseMode.Lower, keepDiacritics: false, keepPunctuations: false, keepNumbers: false)
    .Append(mlContext.Transforms.Text.TokenizeIntoWords(outputColumnName: "Tokens", inputColumnName: "TextNorm"))
    .Append(mlContext.Transforms.Text.RemoveDefaultStopWords(outputColumnName: "TokensClean", inputColumnName: "Tokens",
        language: StopWordsRemovingEstimator.Language.English))
    .Append(mlContext.Transforms.Text.FeaturizeText(outputColumnName: "FeaturesRaw", inputColumnName: "TokensClean"));

// Обучаем на полном корпусе
var textTransformer = textPipeline.Fit(fullData);
var trainFeats = textTransformer.Transform(trainData);
var testFeats  = textTransformer.Transform(testData);

// Размерность и разреженность
int featSize = ((VectorDataViewType)trainFeats.Schema["FeaturesRaw"].Type).Size;
long mTr = trainList.Count, mTe = testList.Count;

long CountNonZeros(IDataView dv)
{
    long nnz = 0;
    var col = dv.Schema["FeaturesRaw"];
    using var cursor = dv.GetRowCursor(new[] { col });
    var accessor = cursor.GetGetter<VBuffer<float>>(col);
    VBuffer<float> v = default;
    while (cursor.MoveNext()) { accessor(ref v); nnz += v.GetValues().Length; }
    return nnz;
}

long nnzTr = CountNonZeros(trainFeats);
long nnzTe = CountNonZeros(testFeats);
long totalTr = mTr * featSize;
long totalTe = mTe * featSize;

Console.WriteLine($"TF-IDF: train shape = ({mTr}, {featSize}), test shape = ({mTe}, {featSize})");
Console.WriteLine($"Train sparsity: {100.0 * nnzTr / totalTr:F3}%");
Console.WriteLine($"Test  sparsity: {100.0 * nnzTe / totalTe:F3}%");


### 4) Снижение размерности (k=500) и L2-нормализация

In [ ]:
int rank = 500;
var lsaPipeline =
    mlContext.Transforms.ProjectToPrincipalComponents(outputColumnName: "FeaturesPca", inputColumnName: "FeaturesRaw", rank: rank, ensureZeroMean: false)
    .Append(mlContext.Transforms.NormalizeLpNorm(outputColumnName: "Features", inputColumnName: "FeaturesPca"));

// Обучаем на train
var lsaTransformer = lsaPipeline.Fit(trainFeats);
var trainLsa = lsaTransformer.Transform(trainFeats);
var testLsa  = lsaTransformer.Transform(testFeats);

Console.WriteLine($"LSA features: train=({mTr}, {rank}), test=({mTe}, {rank})");


### 4.1) Накопленный процент информации (аналог explained_variance_ratio_)

In [ ]:
// Оценим долю объяснённой дисперсии по компонентам PCA через дисперсии проекций
public class PcaOnly { public VBuffer<float> FeaturesPca { get; set; } }
var pcaRows = mlContext.Data.CreateEnumerable<PcaOnly>(trainLsa, reuseRowObject: false);
double[] sum = new double[rank];
double[] sumsq = new double[rank];
long nObs = 0;
foreach (var row in pcaRows)
{
    var vb = row.FeaturesPca;
    var dense = new float[vb.Length]; vb.CopyTo(dense);
    int k = Math.Min(rank, dense.Length);
    for (int j = 0; j < k; j++) { double v = dense[j]; sum[j] += v; sumsq[j] += v * v; }
    nObs++;
}
double[] varComp = new double[rank];
double totalVar = 0;
for (int j = 0; j < rank; j++) { double mean = sum[j] / Math.Max(1, nObs); double v = sumsq[j] / Math.Max(1, nObs) - mean * mean; if (v < 0) v = 0; varComp[j] = v; totalVar += v; }
double[] evr = new double[rank];
double[] evrCumulative = new double[rank];
double acc = 0;
for (int j = 0; j < rank; j++) { evr[j] = totalVar > 0 ? varComp[j] / totalVar : 0; acc += evr[j]; evrCumulative[j] = acc * 100.0; }

var xsEVR = Enumerable.Range(1, rank).Select(i => (double)i).ToArray();
var ysEVR = evrCumulative;
var pltElbow = new ScottPlot.Plot();
pltElbow.Add.Scatter(xsEVR, ysEVR);
pltElbow.Axes.Bottom.Label.Text = "Number of Topics";
pltElbow.Axes.Left.Label.Text = "Cumulative % of Information Retained";
pltElbow.Title("Information Retained");
pltElbow.Axes.Left.Min = 0;
pltElbow.Axes.Left.Max = 100;
// сетка включена по умолчанию в ScottPlot 5
pltElbow


### 5) Классификация (логистическая регрессия) + метрики для train/test

In [ ]:
var trainer = mlContext.BinaryClassification.Trainers.LbfgsLogisticRegression(labelColumnName: "Label", featureColumnName: "Features");
var model = trainer.Fit(trainLsa);

var predTrain = model.Transform(trainLsa);
var predTest  = model.Transform(testLsa);

var mTrain = mlContext.BinaryClassification.Evaluate(predTrain, labelColumnName: "Label");
var mTest  = mlContext.BinaryClassification.Evaluate(predTest,  labelColumnName: "Label");

Console.WriteLine($"Train accuracy: {mTrain.Accuracy:F4}");
Console.WriteLine($"Train F1: {mTrain.F1Score:F4}");

Console.WriteLine($"Test accuracy:  {mTest.Accuracy:F4}");
Console.WriteLine($"Test F1:       {mTest.F1Score:F4}");
Console.WriteLine($"Test AUC:      {mTest.AreaUnderRocCurve:F4}");

Console.WriteLine("\nPer-class (test):");
Console.WriteLine($"  Positive: precision={mTest.PositivePrecision:F3}, recall={mTest.PositiveRecall:F3}");
Console.WriteLine($"  Negative: precision={mTest.NegativePrecision:F3}, recall={mTest.NegativeRecall:F3}");


### 6.1) Примеры: исходный, преобразованный текст, истинная/предсказанная метка

In [ ]:
public class PredOut { public bool PredictedLabel { get; set; } public float Probability { get; set; } }
var preds = mlContext.Data.CreateEnumerable<PredOut>(predTest, reuseRowObject: false).ToArray();
int show = Math.Min(7, Math.Min(testList.Count, preds.Length));
for (int i = 0; i < show; i++)
{
    var ex = testList[i];
    var pr = preds[i];
    Console.WriteLine($"Original sentence: {ex.OriginalText}");
    Console.WriteLine($"Transformed sentence: {ex.Text}");
    Console.WriteLine($"Actual label: {(ex.Label ? 1 : -1)}");
    Console.WriteLine($"Predicted label: {(pr.PredictedLabel ? 1 : -1)}");
    Console.WriteLine(new string('-', 50));
}


### 6) Визуализация компонент (1 vs 5 и 2 vs 4)

In [ ]:
public class PcaOut { public bool Label { get; set; } public VBuffer<float> FeaturesPca { get; set; } }
var rows = mlContext.Data.CreateEnumerable<PcaOut>(trainLsa, reuseRowObject: false).Take(5000).ToArray();

double[] xs1 = new double[rows.Length]; double[] ys1 = new double[rows.Length]; bool[] labs = new bool[rows.Length];
double[] xs2 = new double[rows.Length]; double[] ys2 = new double[rows.Length];
for (int i = 0; i < rows.Length; i++)
{
    var v = rows[i].FeaturesPca;
    var dense = new float[v.Length]; v.CopyTo(dense);
    labs[i] = rows[i].Label;
    if (dense.Length >= 6) { xs1[i] = dense[0]; ys1[i] = dense[4]; xs2[i] = dense[1]; ys2[i] = dense[3]; }
}

int[] idxPos = Enumerable.Range(0, rows.Length).Where(i => labs[i]).ToArray();
int[] idxNeg = Enumerable.Range(0, rows.Length).Where(i => !labs[i]).ToArray();

var plt1 = new ScottPlot.Plot();
var pos1 = plt1.Add.Scatter(idxPos.Select(i=>xs1[i]).ToArray(), idxPos.Select(i=>ys1[i]).ToArray());
pos1.LegendText = "positive";
var neg1 = plt1.Add.Scatter(idxNeg.Select(i=>xs1[i]).ToArray(), idxNeg.Select(i=>ys1[i]).ToArray());
neg1.LegendText = "negative";
plt1.Title("Компоненты: 1 vs 5");
plt1.Axes.Bottom.Label.Text = "Компонента 1";
plt1.Axes.Left.Label.Text = "Компонента 5";
plt1.Legend.IsVisible = true;
plt1


In [ ]:
var plt2 = new ScottPlot.Plot();
var pos2 = plt2.Add.Scatter(idxPos.Select(i=>xs2[i]).ToArray(), idxPos.Select(i=>ys2[i]).ToArray());
pos2.LegendText = "positive";
var neg2 = plt2.Add.Scatter(idxNeg.Select(i=>xs2[i]).ToArray(), idxNeg.Select(i=>ys2[i]).ToArray());
neg2.LegendText = "negative";
plt2.Title("Компоненты: 2 vs 4");
plt2.Axes.Bottom.Label.Text = "Компонента 2";
plt2.Axes.Left.Label.Text = "Компонента 4";
plt2.Legend.IsVisible = true;
plt2
